In [2]:
import pandas as pd
from tqdm.std import tqdm
from openai import OpenAI
import re
import os
import json

In [33]:
input_data = "example_input_files/mathematical_reasoning_0126.csv"
output_data = "example_output_files/mathematical_reasoning_0126.xlsx"

In [37]:
## Open AI API
os.environ["OPENAI_API_KEY"] = "your open ai api"

## common functions

In [5]:
def to_pascal_case(phrase):
    # Remove parentheses and the content inside them, e.g., (Unaudited)
    phrase = re.sub(r'\s*\([^)]*\)', '', phrase)
    
    # Split on non-alphanumeric characters
    words = re.split(r'[^a-zA-Z0-9]', phrase)
    
    # Capitalize each word and join
    return ''.join(word.capitalize() for word in words if word)

In [6]:
def read_xml(xml_path):
    with open(xml_path, "r", encoding="utf-8") as f:
        content = f.read()

        return content

In [36]:
"""
    This function is different a bit among different tasks, here is for the mathematical reasoning task
"""
def construct_annotation(df, results, task_name="semantic_matching"):
    """
        the options of task_name are semantic_matching, relationship_extraction, and mathematical_reasoning
    """
    new_df = df.copy(deep=True)

    schema = []
    presentation = []
    calculation = []
    definition = []
    label = []
    instance = []
    us_gaap = []
    true_answer = []
    segmentation_path = []
    parse_ok = []
    input = []
    outputByLLM = []

    
    for res in results:
        segmentation_name = res.get("segmentation_name")
        folder_name = res.get("folder_name")

        prefix_path = os.path.join(f"second_round_filter/segmentation_filing/{task_name}", folder_name)
        real_segmentation_names = os.listdir(prefix_path)
        potential_seg_names = []

        ## 
        for seg_name in real_segmentation_names:
            normalized_segmentation_name = segmentation_name.lower()
            normalized_seg_name = seg_name.lower()
            if (normalized_segmentation_name in normalized_seg_name) or (normalized_seg_name in normalized_segmentation_name):
                potential_seg_names.append(seg_name)
        
        parse = False
        cal = None
        pre = None
        defn = None
        lab = None
        ins = None
        sch = None
        seg_path = None
        
        ##special for mathematical reasoning task
        answers = {"extracted_value": res["error_tags"].get("reported_value"), "calculated_value": res["error_tags"].get("correct_value"), "is_correct": "Yes"}
        special_inputs = {"tag": res["error_tags"].get("tag"), "period": res["error_tags"].get("period")}

        for segmentation_name_ in potential_seg_names:
            
            full_path = os.path.join(prefix_path, segmentation_name_)
    
            temp_cal = None
            temp_pre = None
            temp_defn = None
            temp_lab = None
            temp_ins = None
            temp_sch = None
            
            file_list = os.listdir(full_path)
            
            for file_name in file_list:
                if file_name.endswith("cal.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_cal = read_xml(final_path)
                elif file_name.endswith("pre.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_pre = read_xml(final_path)
                elif file_name.endswith("def.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_defn = read_xml(final_path)
                elif file_name.endswith("lab.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_lab = read_xml(final_path)
                elif file_name.endswith("htm.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_ins = read_xml(final_path)
                elif file_name.endswith("xsd"):
                    final_path = os.path.join(full_path,file_name)
                    temp_sch = read_xml(final_path)
                else:
                    continue

            is_appear = True
            ## check the instance document
            check_tag = special_inputs.get("tag")
            
            try:
                if check_tag in temp_ins:
                    pass
                elif check_tag.replace(":","") in temp_ins:
                    pass
                elif check_tag.replace(":","_") in temp_ins:
                    pass
                else:
                    is_appear = False
               
                if is_appear:
                    parse = True
                    cal = temp_cal
                    pre = temp_pre
                    defn = temp_defn
                    lab = temp_lab
                    ins = temp_ins
                    sch = temp_sch
                    seg_path = os.path.join(folder_name, segmentation_name_)
                    break
            except Exception as e:
                print(f"Errors {e}")
            
        calculation.append(cal)
        presentation.append(pre)
        definition.append(defn)
        label.append(lab)
        instance.append(ins)
        schema.append(sch)
        true_answer.append(answers)
        us_gaap.append(None)
        parse_ok.append(parse)
        segmentation_path.append(seg_path)
        input.append(special_inputs)
        outputByLLM.append(res)
        
     
        
    new_df["schema"] = schema
    new_df["presentation"] = presentation
    new_df["calculation"] = calculation
    new_df["definition"] = definition
    new_df["label"] = label
    new_df["instance"] = instance
    new_df["us_gaap"] = us_gaap
    new_df["input"] = input
    new_df["true_answer"] = true_answer
    new_df["segmentation_path"] = segmentation_path
    new_df["parse_successful"] = parse_ok
    new_df["LLM_Output"] = outputByLLM

    return new_df

## load dqc messages for relationship extraction task

In [7]:
mr_126_df = pd.read_csv(input_data)

In [8]:
mr_126_df

,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,report.base-taxonomy,report.creation-software,report.document-type,report.entry-url,report.filing-date,report.filing-year,report.id,report.sec-url,report.sic-code,folder_name
0,DQC.US.0126.9590,"2024-11-18 06:31:56,665 [DQC.US.0126.9590] The...",2024-11-18,ERROR,DQC,126,55234,KENILWORTH SYSTEMS CORPORATION,2024-11-14 11:53:00,0001477932-24-007166,US GAAP 2024,XBRL Document Created with XBRLMaster,10-Q,http://www.sec.gov/Archives/edgar/data/55234/0...,2024-11-14,2024,781151,https://www.sec.gov/Archives/edgar/data/55234/...,7990,10q-kenilworth-20240930
1,DQC.US.0126.9590,"2024-01-24 06:46:46,709 [DQC.US.0126.9590] The...",2024-01-24,ERROR,DQC,126,1164256,"DAYBREAK OIL AND GAS, INC.",2024-01-23 17:44:00,0001515971-24-000014,US GAAP 2022,Novaworks,10-K,http://www.sec.gov/Archives/edgar/data/1164256...,2024-01-24,2024,669051,https://www.sec.gov/Archives/edgar/data/116425...,1311,10k-dbrm-20230228
2,DQC.US.0126.9590,"2022-11-21 08:45:40,151 [DQC.US.0126.9590] The...",2022-11-21,ERROR,DQC,126,1389067,"ARtelligence Holdings, Inc.",2022-11-21 08:12:00,0001477932-22-008734,US GAAP 2022,XBRL Document Created with XBRLMaster,10-Q,http://www.sec.gov/Archives/edgar/data/1389067...,2022-11-21,2022,530903,https://www.sec.gov/Archives/edgar/data/138906...,7372,10q-ttcm-20220930
3,DQC.US.0126.9590,"2024-03-14 17:34:25,357 [DQC.US.0126.9590] The...",2024-03-14,ERROR,DQC,126,55234,KENILWORTH SYSTEMS CORPORATION,2024-03-14 14:31:00,0001477932-24-001211,US GAAP 2023,XBRL Document Created with XBRLMaster,10-K,http://www.sec.gov/Archives/edgar/data/55234/0...,2024-03-14,2024,685353,https://www.sec.gov/Archives/edgar/data/55234/...,7990,10k-kens-20231231
4,DQC.US.0126.9590,"2022-03-09 20:00:03,338 [DQC.US.0126.9590] The...",2022-03-10,ERROR,DQC,126,1383088,CYTTA CORP.,2022-01-12 13:46:00,0001477932-22-000209,US GAAP 2020,XBRL Document Created with XBRLMaster,10-K,http://www.sec.gov/Archives/edgar/data/1383088...,2022-01-12,2022,411725,https://www.sec.gov/Archives/edgar/data/138308...,7372,10k-cyca-20210930
5,DQC.US.0126.9590,"2022-12-19 13:18:17,663 [DQC.US.0126.9590] The...",2022-12-19,ERROR,DQC,126,748268,"Red Cat Holdings, Inc.",2022-12-15 16:28:00,0001554795-22-000405,US GAAP 2022,Novaworks,10-Q,http://www.sec.gov/Archives/edgar/data/748268/...,2022-12-15,2022,535639,https://www.sec.gov/Archives/edgar/data/748268...,7372,10q-rcat-20221031
6,DQC.US.0126.9590,"2022-03-04 06:33:27,427 [DQC.US.0126.9590] The...",2022-03-04,ERROR,DQC,126,1576169,"BENEFITFOCUS, INC.",2022-03-03 19:40:00,0001564590-22-008597,US GAAP 2021,DFIN ActiveDisclosure(SM) Inline XBRL Document...,10-K,http://www.sec.gov/Archives/edgar/data/1576169...,2022-03-04,2022,449765,https://www.sec.gov/Archives/edgar/data/157616...,7372,10k-bnft-20211231
7,DQC.US.0126.9590,"2022-03-09 20:00:03,339 [DQC.US.0126.9590] The...",2022-03-10,ERROR,DQC,126,1383088,CYTTA CORP.,2022-01-12 13:46:00,0001477932-22-000209,US GAAP 2020,XBRL Document Created with XBRLMaster,10-K,http://www.sec.gov/Archives/edgar/data/1383088...,2022-01-12,2022,411725,https://www.sec.gov/Archives/edgar/data/138308...,7372,10k-cyca-20210930
8,DQC.US.0126.9590,"2022-11-23 14:03:13,911 [DQC.US.0126.9590] The...",2022-11-23,ERROR,DQC,126,1710495,PINEAPPLE EXPRESS CANNABIS COMPANY,2022-11-23 06:16:00,0001710495-22-000011,US GAAP 2022,Novaworks,10-Q,http://www.sec.gov/Archives/edgar/data/1710495...,2022-11-23,2022,531648,https://www.sec.gov/Archives/edgar/data/171049...,7372,10q-mnao-20221031
9,DQC.US.0126.9590,"2022-03-09 20:10:36,330 [DQC.US.0126.9590] The...",2022-03-10,ERROR,DQC,126,712515,ELECTRONIC ARTS INC.,2022-02-08 16:06:00,0000712515-22-000003,US GAAP 2021,XBRL Document Created with Wdesk from Workiva,10-Q,http://www.sec.gov/Archives/edgar/data/712515/...,2022-02-08,2022,416925,https://www.sec.gov/Archives/edgar/data/712515...,7372,10q-ea-20211231


## LLM judge assisstant

In [9]:
Client = OpenAI()

In [11]:
mr_126_system_prompt = """You are given a DQC validation message. Your task is to extract the following key information:

1. **Statement Name**: Extract the name of the financial statement mentioned in the message. If it appears in a phrase like "000004 - Statement - CONSOLIDATED STATEMENTS OF OPERATIONS (Unaudited)", ignore any numeric prefixes and the phrase "- Statement -". Only return the clean name of the statement, such as "CONSOLIDATED STATEMENTS OF OPERATIONS (Unaudited)".

2. **Erroneous Total Element Tag Information**:
   Extract detailed information about the total element (`us-gaap:` tag) that is reported incorrectly. Specifically, you should extract:

   - `tag`: The total element tag, as indicated by the **“Total Element”** field in the message.
   - `period`: The reporting period associated with this tag, as indicated by the **“Total period”** field.
   - `reported_value`: The value that was reported in the filing, taken from the **“Total Value”** field.
   - `correct_value`: The expected value, i.e., the **sum of the child components** as defined in the calculation linkbase.
     - This is usually explicitly stated in a sentence like: *“The sum of these child components is...”*
     - You may cross-validate this by summing the listed child component values in the message if provided.
   - `reason`: Provide a concise explanation of the error — why the reported value is considered incorrect — based directly on the message content.

Only include this tag if all of the following are true:
- The message identifies a total element with incorrect calculation
- Both reported and correct values are stated or inferable
- The error occurs in a specified reporting period

Return your result as structured JSON in the following format:
```json
{
  "statement_name": "...",
  "error_tags": 
    {
      "tag": "...",
      "period": "...",
      "reported_value": "...",
      "correct_value": "...",
      "reason": "..."
    }
}
"""

In [12]:
def get_response(dqc_message):
    
    completion = Client.chat.completions.create(
        model="gpt-4.1-mini",
        response_format={
            "type": "json_object"
        },
        messages=[
            {"role": "system", "content": mr_126_system_prompt},
            {"role": "user", "content": f"DQC validation message: {dqc_message} \n Output:"}
        ]
    )
    return completion.choices[0].message.content

In [14]:
res = []
for _, row in tqdm(mr_126_df.iterrows()):
    ## load each dqc message
    message = row["assertion.detail"]
    response = get_response(message)
    response = json.loads(response)

    ## load the folder name which will be used to obtain candidate segmentation folders
    folder_name = row["folder_name"]

    ## covert the extracted statement name in the message to PascalCase format as the segmentation name
    segmentation_name = to_pascal_case(response.get("statement_name"))

    ## record the folder name
    response["folder_name"] = folder_name
    ## record the extracted segmenataion name
    response["segmentation_name"] = segmentation_name
    res.append(response)

10it [00:33,  3.31s/it]


In [19]:
res[4]

{'statement_name': 'Statements of Cash Flows',
 'error_tags': {'tag': 'us-gaap:NetCashProvidedByUsedInOperatingActivities',
  'period': '2020-10-01 to 2021-09-30',
  'reported_value': '-1,313,536',
  'correct_value': '-1,212,736',
  'reason': 'The reported total does not match the sum of its child components defined in the calculation linkbase. The total reported is -1,313,536 while the sum of the child components is -1,212,736, indicating an inconsistency in the reported calculation.'},
 'folder_name': '10k-cyca-20210930',
 'segmentation_name': 'StatementsOfCashFlows'}

In [30]:
"""
    the options of task_name are semantic_matching, relationship_extraction, and mathematical_reasoning
"""
new_mr_126_df = construct_annotation(mr_126_df, res, task_name="mathematical_reasoning")

Errors argument of type 'NoneType' is not iterable
Errors argument of type 'NoneType' is not iterable


In [31]:
new_mr_126_df

,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,...,calculation,definition,label,instance,us_gaap,input,true_answer,segmentation_path,parse_successful,LLM_Output
0,DQC.US.0126.9590,"2024-11-18 06:31:56,665 [DQC.US.0126.9590] The...",2024-11-18,ERROR,DQC,126,55234,KENILWORTH SYSTEMS CORPORATION,2024-11-14 11:53:00,0001477932-24-007166,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,"{'tag': 'us-gaap:OperatingIncomeLoss', 'period...","{'extracted_value': '-247,587', 'calculated_va...",10q-kenilworth-20240930/ConsolidatedStatements...,True,{'statement_name': 'CONSOLIDATED STATEMENTS OF...
1,DQC.US.0126.9590,"2024-01-24 06:46:46,709 [DQC.US.0126.9590] The...",2024-01-24,ERROR,DQC,126,1164256,"DAYBREAK OIL AND GAS, INC.",2024-01-23 17:44:00,0001515971-24-000014,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,{'tag': 'us-gaap:NetCashProvidedByUsedInInvest...,"{'extracted_value': '-386,160', 'calculated_va...",10k-dbrm-20230228/StatementsOfCashFlows,True,"{'statement_name': 'Statements of Cash Flows',..."
2,DQC.US.0126.9590,"2022-11-21 08:45:40,151 [DQC.US.0126.9590] The...",2022-11-21,ERROR,DQC,126,1389067,"ARtelligence Holdings, Inc.",2022-11-21 08:12:00,0001477932-22-008734,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,{'tag': 'us-gaap:CashCashEquivalentsRestricted...,"{'extracted_value': '-115,117', 'calculated_va...",10q-ttcm-20220930/ConsolidatedStatementsOfCash...,True,{'statement_name': 'CONSOLIDATED STATEMENTS OF...
3,DQC.US.0126.9590,"2024-03-14 17:34:25,357 [DQC.US.0126.9590] The...",2024-03-14,ERROR,DQC,126,55234,KENILWORTH SYSTEMS CORPORATION,2024-03-14 14:31:00,0001477932-24-001211,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,{'tag': 'us-gaap:NetCashProvidedByUsedInOperat...,"{'extracted_value': '-118,456', 'calculated_va...",10k-kens-20231231/ConsolidatedStatementsOfCash...,True,{'statement_name': 'CONSOLIDATED STATEMENTS OF...
4,DQC.US.0126.9590,"2022-03-09 20:00:03,338 [DQC.US.0126.9590] The...",2022-03-10,ERROR,DQC,126,1383088,CYTTA CORP.,2022-01-12 13:46:00,0001477932-22-000209,...,None,None,None,None,None,{'tag': 'us-gaap:NetCashProvidedByUsedInOperat...,"{'extracted_value': '-1,313,536', 'calculated_...",None,False,"{'statement_name': 'Statements of Cash Flows',..."
5,DQC.US.0126.9590,"2022-12-19 13:18:17,663 [DQC.US.0126.9590] The...",2022-12-19,ERROR,DQC,126,748268,"Red Cat Holdings, Inc.",2022-12-15 16:28:00,0001554795-22-000405,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,"{'tag': 'us-gaap:AssetsNoncurrent', 'period': ...","{'extracted_value': '30,227,410', 'calculated_...",10q-rcat-20221031/BalanceSheets,True,{'statement_name': 'Consolidated Balance Sheet...
6,DQC.US.0126.9590,"2022-03-04 06:33:27,427 [DQC.US.0126.9590] The...",2022-03-04,ERROR,DQC,126,1576169,"BENEFITFOCUS, INC.",2022-03-03 19:40:00,0001564590-22-008597,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,"{'tag': 'us-gaap:StockholdersEquity', 'period'...","{'extracted_va

In [32]:
new_mr_126_df.to_excel(output_data,index=False)